# 🚀 Employment Prediction Hackathon — R Testing Kit
This notebook walks you from raw data to your submission, step by step data
**The goal:** for each person in Round 9, predict the *probability* that they are **employed (1)** vs **unemployed (0)**.
Your submissions are scored on **AUC**, so always submit a probability between 0 and 1 — never just a hard 0/1.

## 🛠️ Step 1: Load tools and Data

Before we begin exploring and modelling the data, let’s load the required R packages and import the training and testing datasets. This gives us everything we need to start building our employment prediction model.



In [1]:
# install.packages(c('scales','pROC'))  # uncomment on first run if needed
suppressMessages({
  library(dplyr)
  library(ggplot2)
  library(scales)
})

# Load the competition datasets (adjust the paths to match your Kaggle input folder)
data_dir <- '../../assets/dataset'
train_data <- read.csv(file.path(data_dir, 'train.csv'), stringsAsFactors = FALSE)
test_data  <- read.csv(file.path(data_dir, 'test.csv'),  stringsAsFactors = FALSE)

cat('Training rows:', nrow(train_data), '| columns:', ncol(train_data), '\n')
cat('Testing rows :', nrow(test_data),  '| columns:', ncol(test_data),  '\n')

# The target column in train_data is `employed_status` (1 = employed, 0 = unemployed).
# It is NOT present in test_data — that is what you predict.


Training rows: 16719 | columns: 33 
Testing rows : 3230 | columns: 32 


## Step 2: Statistical Testing & Feature Selection
Before modelling, look at what the data actually contains. We use different tests to determine what we can find out about the data and how variables interact with dependable variable.
> **Sizing tip:** on Kaggle/Jupyter, control chart size with `options(repr.plot.width = ..., repr.plot.height = ...)`
> *before* the plot — not with fixed panel sizes. We set sensible sizes per chart below so nothing gets cropped.


In [2]:
# A tidy plotting frame with readable labels
plot_df <- train_data %>%
  filter(!is.na(employed_status)) %>%
  mutate(Status = factor(employed_status, levels = c(0, 1),
                         labels = c('Unemployed', 'Employed')))

pal <- c('Unemployed' = '#E15759', 'Employed' = '#59A14F')

# A reusable, clean theme
theme_hack <- theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = 'bold'),
        plot.subtitle = element_text(color = 'grey35'),
        legend.position = 'top')


### 2.1 — ANOVA Table
Test whether the mean of a numerical variable differs significantly between employed and unemployed groups.

In [4]:
# ============================================
# 2.1 ANOVA Table
# ============================================

# Create employment groups for the ANOVA
train_data$employed_status_factor <- factor(
  train_data$employed_status,
  levels = c(0, 1),
  labels = c("Unemployed", "Employed")
)

# Variables to test
numeric_vars <- c(
  "age",
  "days_since_last_obs",
  "total_historical_rounds",
  "tenure_lag",
  "work_readiness_score"
)

# Keep only variables that exist in the dataset
numeric_vars <- numeric_vars[
  numeric_vars %in% names(train_data)
]

# Run ANOVA for each numerical variable
for (variable in numeric_vars) {

  formula <- as.formula(
    paste(variable, "~ employed_status_factor")
  )

  anova_model <- aov(
    formula,
    data = train_data
  )

  cat("\n========================================\n")
  cat("ANOVA Test:", variable, "\n")
  cat("========================================\n")

  print(summary(anova_model))
}


ANOVA Test: age 
                          Df Sum Sq Mean Sq F value   Pr(>F)    
employed_status_factor     1    897   896.9   48.73 3.05e-12 ***
Residuals              16645 306379    18.4                     
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
72 observations deleted due to missingness

ANOVA Test: days_since_last_obs 
                         Df   Sum Sq Mean Sq F value Pr(>F)
employed_status_factor    1     1361    1361   0.138   0.71
Residuals              5260 51709437    9831               
11457 observations deleted due to missingness

ANOVA Test: total_historical_rounds 
                          Df Sum Sq Mean Sq F value Pr(>F)    
employed_status_factor     1     34   34.36   79.58 <2e-16 ***
Residuals              16651   7189    0.43                   
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
66 observations deleted due to missingness

ANOVA Test: tenure_lag 
                         Df    Sum Sq  Mean Sq F value 

**What it means:** the classes are imbalanced (~1 in 3 employed). A model that predicts 'unemployed' for everyone would look ~68% 'accurate' but is useless — this is exactly why the competition uses **AUC**, which rewards ranking people by their probability of employment rather than raw accuracy.

## Generic Anova table 

In [ ]:
# ============================================
# 2.1 ANOVA Table
# ============================================

# Create employment groups for the ANOVA
train_data$employed_status_factor <- factor(
  train_data$employed_status,
  levels = c(0, 1),
  labels = c("Unemployed", "Employed")
)

# Variables to test
numeric_vars <- c(
  "age",
  "days_since_last_obs",
  "total_historical_rounds",
  "tenure_lag",
  "work_readiness_score"
)

# Keep only variables that exist in the dataset
numeric_vars <- numeric_vars[
  numeric_vars %in% names(train_data)
]

# Run ANOVA for each numerical variable
for (variable in numeric_vars) {

  formula <- as.formula(
    paste(variable, "~ employed_status_factor")
  )

  anova_model <- aov(
    formula,
    data = train_data
  )

  cat("\n========================================\n")
  cat("ANOVA Test:", variable, "\n")
  cat("========================================\n")

  print(summary(anova_model))
}


ANOVA Test: age 
                          Df Sum Sq Mean Sq F value   Pr(>F)    
employed_status_factor     1    897   896.9   48.73 3.05e-12 ***
Residuals              16645 306379    18.4                     
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
6 observations deleted due to missingness

ANOVA Test: days_since_last_obs 
                         Df   Sum Sq Mean Sq F value Pr(>F)
employed_status_factor    1     1361    1361   0.138   0.71
Residuals              5260 51709437    9831               
11391 observations deleted due to missingness

ANOVA Test: total_historical_rounds 
                          Df Sum Sq Mean Sq F value Pr(>F)    
employed_status_factor     1     34   34.36   79.58 <2e-16 ***
Residuals              16651   7189    0.43                   
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

ANOVA Test: tenure_lag 
                         Df    Sum Sq  Mean Sq F value  Pr(>F)    
employed_status_factor    1 1.69

### 2.2 — Stepwise Regression
Use stepwise selection to identify variables that contribute most strongly to the model.

In [5]:
# ============================================
# 2.2 Stepwise Logistic Regression
# ============================================

# Candidate predictor variables
candidate_vars <- c(
  "age",
  "gender",
  "race",
  "sa_citizen",
  "province",
  "education_level",
  "education_schooling_grade_twelve_equivalent",
  "institution_type",
  "school_quintile",
  "education_field",
  "employed_lag",
  "status_lag",
  "status_broad_lag",
  "tenure_lag",
  "total_historical_rounds",
  "days_since_last_obs",
  "work_readiness_score"
)

# Keep only columns that exist
candidate_vars <- candidate_vars[
  candidate_vars %in% names(train_data)
]

# --------------------------------------------
# Create modelling dataset
# --------------------------------------------

model_vars <- c(
  "employed_status",
  candidate_vars
)

stepwise_data <- train_data[, model_vars, drop = FALSE]

# Remove rows with missing values
stepwise_data <- na.omit(stepwise_data)

cat("Rows available for modelling:",
    nrow(stepwise_data), "\n")

# --------------------------------------------
# Remove predictors with only one level
# AFTER removing missing values
# --------------------------------------------

valid_vars <- c()

cat("\nChecking predictor variables:\n")

for (variable in candidate_vars) {

  # Number of unique values remaining
  # after NA rows have been removed
  number_of_levels <- length(
    unique(stepwise_data[[variable]])
  )

  if (number_of_levels >= 2) {

    valid_vars <- c(valid_vars, variable)

    cat("✓", variable,
        "- ", number_of_levels, " levels\n")

  } else {

    cat("✗", variable,
        "- ONLY ", number_of_levels,
        " level; removed\n")
  }
}

# --------------------------------------------
# Check target variable
# --------------------------------------------

target_levels <- unique(stepwise_data$employed_status)

cat("\nEmployment status levels:\n")
print(target_levels)

if (length(target_levels) < 2) {

  stop(
    "Stepwise regression cannot run because ",
    "employed_status has fewer than 2 levels."
  )
}

# --------------------------------------------
# Convert categorical variables to factors
# --------------------------------------------

categorical_vars <- c(
  "gender",
  "race",
  "sa_citizen",
  "province",
  "education_level",
  "education_schooling_grade_twelve_equivalent",
  "institution_type",
  "education_field",
  "employed_lag",
  "status_lag",
  "status_broad_lag"
)

categorical_vars <- intersect(
  categorical_vars,
  valid_vars
)

for (variable in categorical_vars) {
  stepwise_data[[variable]] <- factor(
    stepwise_data[[variable]]
  )
}

# --------------------------------------------
# Build logistic regression formula
# --------------------------------------------

full_formula <- as.formula(
  paste(
    "employed_status ~",
    paste(valid_vars, collapse = " + ")
  )
)

cat("\n========================================\n")
cat("Full Logistic Regression Model\n")
cat("========================================\n")

print(full_formula)

# --------------------------------------------
# Fit full model
# --------------------------------------------

full_model <- glm(
  full_formula,
  data = stepwise_data,
  family = binomial()
)

summary(full_model)

# --------------------------------------------
# Stepwise selection
# --------------------------------------------

cat("\n========================================\n")
cat("Stepwise Regression\n")
cat("========================================\n")

stepwise_model <- step(
  full_model,
  direction = "both",
  trace = TRUE
)

# --------------------------------------------
# Final model
# --------------------------------------------

cat("\n========================================\n")
cat("Final Stepwise Model\n")
cat("========================================\n")

summary(stepwise_model)

# --------------------------------------------
# Selected variables
# --------------------------------------------

cat("\n========================================\n")
cat("Selected Variables\n")
cat("========================================\n")

print(names(coef(stepwise_model)))

Rows available for modelling: 968 

Checking predictor variables:
✓ age -  19  levels
✓ gender -  2  levels
✓ race -  4  levels
✗ sa_citizen - ONLY  1  level; removed
✓ province -  9  levels
✓ education_level -  4  levels
✗ institution_type - ONLY  1  level; removed
✓ school_quintile -  6  levels
✗ education_field - ONLY  1  level; removed
✓ employed_lag -  2  levels
✓ status_lag -  4  levels
✓ status_broad_lag -  4  levels
✓ tenure_lag -  279  levels
✓ total_historical_rounds -  4  levels
✓ days_since_last_obs -  207  levels
✓ work_readiness_score -  685  levels

Employment status levels:
[1] 1 0

Full Logistic Regression Model
employed_status ~ age + gender + race + province + education_level + 
    school_quintile + employed_lag + status_lag + status_broad_lag + 
    tenure_lag + total_historical_rounds + days_since_last_obs + 
    work_readiness_score



Call:
glm(formula = full_formula, family = binomial(), data = stepwise_data)

Coefficients: (4 not defined because of singularities)
                                         Estimate Std. Error z value Pr(>|z|)
(Intercept)                            -1.975e+00  8.527e-01  -2.317 0.020526
age                                     3.871e-02  2.170e-02   1.784 0.074409
genderMale                              7.914e-01  1.471e-01   5.380 7.44e-08
raceColoured                            8.235e-01  3.723e-01   2.212 0.026988
raceIndian                              1.047e+00  1.459e+00   0.717 0.473150
raceWhite                              -1.333e+01  5.354e+02  -0.025 0.980144
provinceFree State                     -1.405e-01  3.795e-01  -0.370 0.711158
provinceGauteng                        -2.062e-02  2.899e-01  -0.071 0.943287
provinceKwaZulu-Natal                  -4.668e-01  3.234e-01  -1.443 0.148890
provinceLimpopo                        -1.229e-02  3.730e-01  -0.033 0.973714
province


Stepwise Regression
Start:  AIC=1184.98
employed_status ~ age + gender + race + province + education_level + 
    school_quintile + employed_lag + status_lag + status_broad_lag + 
    tenure_lag + total_historical_rounds + days_since_last_obs + 
    work_readiness_score


Step:  AIC=1184.98
employed_status ~ age + gender + race + province + education_level + 
    school_quintile + employed_lag + status_lag + tenure_lag + 
    total_historical_rounds + days_since_last_obs + work_readiness_score


Step:  AIC=1184.98
employed_status ~ age + gender + race + province + education_level + 
    school_quintile + status_lag + tenure_lag + total_historical_rounds + 
    days_since_last_obs + work_readiness_score

                          Df Deviance    AIC
- province                 8   1143.0 1177.0
- education_level          3   1137.7 1181.7
- days_since_last_obs      1   1135.3 1183.3
- total_historical_rounds  1   1135.5 1183.5
- tenure_lag               1   1136.0 1184.0
- work_readiness


Call:
glm(formula = employed_status ~ gender + race + school_quintile + 
    status_lag + work_readiness_score, family = binomial(), data = stepwise_data)

Coefficients:
                                 Estimate Std. Error z value Pr(>|z|)    
(Intercept)                       -0.8771     0.3374  -2.600  0.00933 ** 
genderMale                         0.8001     0.1439   5.560 2.69e-08 ***
raceColoured                       0.9356     0.3559   2.628  0.00858 ** 
raceIndian                         0.7745     1.4385   0.538  0.59033    
raceWhite                        -12.5321   535.4112  -0.023  0.98133    
school_quintile                    0.1126     0.0579   1.945  0.05172 .  
status_lagunemployed              -1.0089     0.2576  -3.916 8.99e-05 ***
status_lagwage and self employed  -0.8504     0.7902  -1.076  0.28187    
status_lagwage employed           -0.1946     0.2894  -0.673  0.50122    
work_readiness_score               0.5447     0.3793   1.436  0.15097    
---
Signif. cod


Selected Variables
 [1] "(Intercept)"                      "genderMale"                      
 [3] "raceColoured"                     "raceIndian"                      
 [5] "raceWhite"                        "school_quintile"                 
 [7] "status_lagunemployed"             "status_lagwage and self employed"
 [9] "status_lagwage employed"          "work_readiness_score"            


In [ ]:
# Variables selected by stepwise regression

selected_variables <- names(
  coef(stepwise_model)
)

selected_variables

[1] "(Intercept)"                      "genderMale"                      
 [3] "raceColoured"                     "raceIndian"                      
 [5] "raceWhite"                        "school_quintile"                 
 [7] "status_lagunemployed"             "status_lagwage and self employed"
 [9] "status_lagwage employed"          "work_readiness_score"

**What it means:** the lag columns are powerful *when present* but empty for most people. Decide deliberately how to handle those NAs — dropping them throws away most of your data. A common trick (used in Step 5) is to treat 'missing history' as its own category rather than discarding the row.

## 📋 2.3 Item Reliability Analysis (IRA)

Item Reliability Analysis evaluates whether multiple items intended to measure the same underlying concept are consistent with one another. In this case, we are interested in whether the available work-readiness assessment items provide a reliable measure of work readiness.

We will use **Cronbach's alpha** to measure internal consistency. A higher alpha generally indicates that the items are measuring a similar underlying construct.

**Your task:** Examine the reliability output and identify whether the items appear to form a reliable scale that could be useful for employment prediction.


In [ ]:
# ============================================
# 2.3 Item Reliability Analysis (IRA)
# ============================================

# Install if needed:
# install.packages("psych")

library(psych)

# ------------------------------------------------
# Find potential work-readiness item variables
# ------------------------------------------------

# Look for columns containing "work_readiness"
work_readiness_items <- names(train_data)[
  grepl("work_readiness", names(train_data), ignore.case = TRUE)
]

cat("Potential work-readiness variables:\n")
print(work_readiness_items)


# ------------------------------------------------
# Remove the overall score if it exists
# ------------------------------------------------

work_readiness_items <- work_readiness_items[
  work_readiness_items != "work_readiness_score"
]

cat("\nItems used for reliability analysis:\n")
print(work_readiness_items)


# ------------------------------------------------
# Check whether enough individual items exist
# ------------------------------------------------

if (length(work_readiness_items) >= 2) {
  
  # Select the item columns
  reliability_data <- train_data[
    , work_readiness_items,
    drop = FALSE
  ]
  
  # Convert columns to numeric where possible
  reliability_data <- data.frame(
    lapply(reliability_data, function(x) {
      as.numeric(as.character(x))
    })
  )
  
  # Run Cronbach's Alpha
  reliability_result <- psych::alpha(
    reliability_data,
    check.keys = TRUE
  )
  
  # Display the complete reliability output
  print(reliability_result)
  
  # Display Cronbach's alpha specifically
  cat("\n----------------------------------------\n")
  cat("Cronbach's Alpha:", reliability_result$total$raw_alpha, "\n")
  cat("----------------------------------------\n")
  
  # Display item-level statistics
  cat("\nItem Statistics:\n")
  print(reliability_result$item.stats)
  
  # Display alpha if each item is removed
  cat("\nAlpha if Item is Deleted:\n")
  print(reliability_result$alpha.drop)
  
} else {
  
  cat("\nNot enough individual work-readiness items were found.\n")
  cat("At least two items are required for Item Reliability Analysis.\n")
  cat("The dataset currently appears to contain only the overall\n")
  cat("work_readiness_score rather than individual assessment items.\n")
}

Installing package into 'C:/Users/27823/AppData/Local/R/win-library/4.4'
(as 'lib' is unspecified)



also installing the dependencies 'mnormt', 'GPArotation'





  There are binary versions available but the source versions are later:
              binary   source needs_compilation
GPArotation 2025.3-1 2026.8-1             FALSE
psych          2.6.3    2.6.5             FALSE

package 'mnormt' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\27823\AppData\Local\Temp\RtmpsLGHpL\downloaded_packages


installing the source packages 'GPArotation', 'psych'



Attaching package: 'psych'


The following objects are masked from 'package:scales':

    alpha, rescale


The following objects are masked from 'package:ggplot2':

    %+%, alpha




Potential work-readiness variables:
[1] "work_readiness_score"

Items used for reliability analysis:
character(0)

Not enough individual work-readiness items were found.
At least two items are required for Item Reliability Analysis.
The dataset currently appears to contain only the overall
work_readiness_score rather than individual assessment items.


**What it means:** someone employed last round is more likely to be employed now than someone who was unemployed. This 'stickiness' is part of the story — but as you'll see, it is not the whole picture, and turning it into leaderboard gains takes care.

### 2.4 — Chi-Square Test
Test whether categorical variables are significantly associated with employed_status.

In [8]:
# 2.4 Chi-Square Tests
# Test whether categorical variables are associated with employment status

categorical_vars <- c(
  "gender",
  "race",
  "sa_citizen",
  "province",
  "education_level",
  "education_schooling_grade_twelve_equiv",
  "institution_type",
  "education_field",
  "status_lag",
  "status_broad_lag"
)

# Keep only variables that exist
categorical_vars <- categorical_vars[
  categorical_vars %in% names(train_data)
]

for (variable in categorical_vars) {
  
  # Create contingency table
  contingency_table <- table(
    train_data[[variable]],
    train_data$employed_status
  )
  
  # Run chi-square test
  chi_test <- chisq.test(contingency_table)
  
  cat("\n========================================\n")
  cat("Chi-Square Test:", variable, "\n")
  cat("========================================\n")
  
  print(contingency_table)
  print(chi_test)
}


Chi-Square Test: gender 
        
            0    1
  Female 7077 2370
  Male   4298 2908

	Pearson's Chi-squared test with Yates' continuity correction

data:  contingency_table
X-squared = 439.46, df = 1, p-value < 2.2e-16



Warning message in chisq.test(contingency_table):
"Chi-squared approximation may be incorrect"



Chi-Square Test: race 
               
                    0     1
                   20    10
  African/Black 10731  4855
  Asian             4     1
  Coloured        544   349
  Indian           47    41
  Other            17     5
  Unspecified       0     1
  White            12    16

	Pearson's Chi-squared test

data:  contingency_table
X-squared = 45.368, df = 7, p-value = 1.16e-07



Warning message in chisq.test(contingency_table):
"Chi-squared approximation may be incorrect"



Chi-Square Test: sa_citizen 
       
            0     1
  FALSE     9     5
  TRUE  11366  5273

	Pearson's Chi-squared test with Yates' continuity correction

data:  contingency_table
X-squared = 0.0013041, df = 1, p-value = 0.9712


Chi-Square Test: province 
               
                   0    1
                  19    9
  Eastern Cape   999  416
  Free State     700  278
  Gauteng       3889 1941
  KwaZulu-Natal 1978  892
  Limpopo       1038  444
  Mpumalanga     847  382
  North West     705  277
  Northern Cape  216   92
  Western Cape   984  547

	Pearson's Chi-squared test

data:  contingency_table
X-squared = 35.427, df = 9, p-value = 5.009e-05



...


Chi-Square Test: education_schooling_grade_twelve_equiv 
       
           0    1
        2508 1139
  0        4    3
  1       33   22
  FALSE  996  403
  TRUE  7834 3711

	Pearson's Chi-squared test

data:  contingency_table
X-squared = 8.9842, df = 4, p-value = 0.0615



Warning message in chisq.test(contingency_table):
"Chi-squared approximation may be incorrect"



Chi-Square Test: institution_type 
                                      
                                          0    1
                                       9318 4386
  Comprehensive University              418  188
  Private College                       278  124
  Private Higher Education Institution  143   73
  Public College                        697  271
  Public University                     200   97
  Tech Accelerator                        2    2
  Trade School                           62   21
  University of Technology              257  116

	Pearson's Chi-squared test

data:  contingency_table
X-squared = 9.8039, df = 8, p-value = 0.2791



Warning message in chisq.test(contingency_table):
"Chi-squared approximation may be incorrect"



Chi-Square Test: education_field 
                                                         
                                                             0    1
                                                          7327 3299
  Accounting and bookkeeping                               104   51
  Administration                                           318  136
  Advertising                                               10    2
  Agriculture                                               74   34
  Aquaculture and fisheries                                  3    0
  Architecture                                               3    0
  Arts and crafts                                           17    8
  Auditing                                                  10    6
  Automation and testing                                     3    3
  Automotive and transport equipment manufacturing           5    3
  Automotive technician                                      2    1
  Aviation and maritime

## 🧪 2.5 Statistical Test Template

Use this template to investigate another relationship in the dataset.

Your test should:

1. State what relationship you are investigating.
2. Identify the variables involved.
3. Run an appropriate statistical test.
4. Record the important statistic and p-value.
5. Decide whether the result is statistically significant.
6. Explain what the result means for employment prediction.

In [9]:
# ============================================
# 2.5 Statistical Test Template
# ============================================

# Variable being investigated
variable_1 <- "REPLACE_WITH_VARIABLE"

# Target variable
variable_2 <- "employed_status"

# Your statistical test here
# Example:
#
# test_result <- chisq.test(
#   table(train_data[[variable_1]], train_data[[variable_2]])
# )
#
# print(test_result)


# Record the important result
# statistic <- ...
# p_value <- ...


# Your conclusion
cat("Variable tested:", variable_1, "\n")
cat("Target variable:", variable_2, "\n")
cat("Test statistic: [SUBMIT VALUE]\n")
cat("P-value: [SUBMIT VALUE]\n")

Variable tested: REPLACE_WITH_VARIABLE 
Target variable: employed_status 
Test statistic: [SUBMIT VALUE]
P-value: [SUBMIT VALUE]


**What it means:** if the boxes overlap heavily, the score alone is a weak separator — and adding weak features to a model can even *hurt* it. Part of the challenge is finding which combinations genuinely carry signal.

2.6 ANOVCA Table

In [10]:
# ============================================
# 2.6 ANOVCA Table
# ============================================

# Create employment groups for the ANOVA
train_data$employed_status_factor <- factor(
  train_data$employed_status,
  levels = c(0, 1),
  labels = c("Unemployed", "Employed")
)

# Variables to test
numeric_vars <- c(
  "age",
  "days_since_last_obs",
  "total_historical_rounds",
  "tenure_lag",
  "work_readiness_score"
)

# Keep only variables that exist in the dataset
numeric_vars <- numeric_vars[
  numeric_vars %in% names(train_data)
]

# Run ANOVA for each numerical variable
for (variable in numeric_vars) {

  formula <- as.formula(
    paste(variable, "~ employed_status_factor")
  )

  anova_model <- aov(
    formula,
    data = train_data
  )

  cat("\n========================================\n")
  cat("ANOVA Test:", variable, "\n")
  cat("========================================\n")

  print(summary(anova_model))
}


ANOVA Test: age 
                          Df Sum Sq Mean Sq F value   Pr(>F)    
employed_status_factor     1    897   896.9   48.73 3.05e-12 ***
Residuals              16645 306379    18.4                     
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
72 observations deleted due to missingness

ANOVA Test: days_since_last_obs 
                         Df   Sum Sq Mean Sq F value Pr(>F)
employed_status_factor    1     1361    1361   0.138   0.71
Residuals              5260 51709437    9831               
11457 observations deleted due to missingness

ANOVA Test: total_historical_rounds 
                          Df Sum Sq Mean Sq F value Pr(>F)    
employed_status_factor     1     34   34.36   79.58 <2e-16 ***
Residuals              16651   7189    0.43                   
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
66 observations deleted due to missingness

ANOVA Test: tenure_lag 
                         Df    Sum Sq  Mean Sq F value 

## Round and Observation Variables

The initial analysis suggests that `lag_round`, `days_since_last_obs`, and `current_round` do not show a statistically significant relationship with employment status. However, these variables should be tested further before being completely excluded from the predictive model.

A particularly strong relationship was observed for `employed_lag`. The very large F-statistic and extremely small p-value indicate a strong association between previous employment status and current employment status. This supports the earlier observation that **someone who was employed previously has a high likelihood of being employed in the future**.

## `employed_lag` and Missing History

`employed_lag` contains a substantial amount of missing data:

* Missing: **11,451 / 16,719 observations**
* Approximately **68.5%** of the values are missing.

This missingness is important because a missing value does not necessarily mean that the participant was unemployed. Instead, it may indicate that **there is no previous observation or employment history available**.

Therefore, rather than treating missing values as ordinary missing data, we can separate the information into two variables:

* **`employed_lag`** — the previous employment status when historical information exists.
* **`employed_lag_missing`** — a binary indicator showing whether previous employment information is missing.

For example:

```text
No previous history
    ↓
employed_lag_missing = 1
employed_lag = NA

Previous history exists
    ↓
employed_lag_missing = 0
employed_lag = 0 or 1
```

This allows the model to distinguish between **previously unemployed**, **previously employed**, and **having no previous employment history**, rather than incorrectly treating missing history as unemployment.


In [18]:
    train_data$employed_lag_missing <- ifelse(
    is.na(train_data$employed_lag), 1, 0)

    
    test_data$employed_lag_missing <- ifelse(
    is.na(test_data$employed_lag), 1, 0)


  

## `employed_lag_missing`

`employed_lag_missing` is a new binary variable that indicates whether the `employed_lag` value is missing.

* **`1`** → `employed_lag` is missing
* **`0`** → `employed_lag` is available

This can be useful for modelling because the **absence of previous employment information may itself contain predictive information**. The model can therefore distinguish between participants with a recorded employment history and those without one.


## `work_readiness_score` and `wr_filled`

`work_readiness_score` and `wr_filled` provide related information about work readiness. Using both directly as predictors may introduce redundancy.

Instead, we can use `wr_filled` as a **binary indicator** showing whether a work-readiness score is available, while `work_readiness_score` contains the actual score when it is present.

This allows us to distinguish between:

* **`wr_filled = 1`** → work-readiness score is available
* **`wr_filled = 0`** → work-readiness score is missing

The missingness itself may contain useful information for predicting employment, so we can test it separately rather than treating the two variables as identical predictors.


In [ ]:
train_data$wr_missing <- ifelse(
  is.na(train_data$work_readiness_score), 1, 0
)

test_data$wr_missing <- ifelse(
  is.na(test_data$work_readiness_score), 1, 0
)

## Investigation of the catergorical variables:

status_lag
status_broad_lag
education_level
education_field
institution_type
gender
province
race



In [17]:
# Chi-squared test for categorical variables
chisq.test(
  table(
    train_data$status_lag,
    train_data$employed_status
  )
)

# Chi-squared test for status_broad_lag vs employed_status
chisq.test(
  table(
    train_data$status_broad_lag,
    train_data$employed_status
  )
)

# Chi-squared test for education_level vs employed_status
chisq.test(
  table(
    train_data$education_level,
    train_data$employed_status
  )
)


	Pearson's Chi-squared test

data:  table(train_data$status_lag, train_data$employed_status)
X-squared = 485.05, df = 5, p-value < 2.2e-16



	Pearson's Chi-squared test

data:  table(train_data$status_broad_lag, train_data$employed_status)
X-squared = 471.29, df = 6, p-value < 2.2e-16


Warning message in chisq.test(table(train_data$education_level, train_data$employed_status)):
"Chi-squared approximation may be incorrect"



	Pearson's Chi-squared test

data:  table(train_data$education_level, train_data$employed_status)
X-squared = 44.366, df = 35, p-value = 0.1333


## Conclusion of the chi-squared test 
| Variable | χ² | df | p-value | Result |
|---|---:|---:|---:|---|
| status_lag | 485.05 | 5 | < 2.2e-16 | Extremely strong |
| status_broad_lag | 471.29 | 6 | < 2.2e-16 | Extremely strong |
| education_level | 44.37 | 35 | 0.1333 | Not significant |


**education_level**
We don't have enough evidence to say education level is associated with employment status in this dataset.

However, I wouldn't immediately remove it. As the p-value is not too high above 0.1 so it still play a factor as logically education still a deciding factor especially NQF levels.

## 🏗️ Step 3: A first baseline model
We predict a binary outcome, so we start with **logistic regression**. For this first attempt we deliberately
This sets a (low) floor — and shows that simply throwing variables at the model is not enough.

In [20]:
# Fill missing work readiness with the training median so every row has a value
wr_median <- median(train_data$work_readiness_score, na.rm = TRUE)
train_data$wr_filled <- ifelse(is.na(train_data$work_readiness_score), wr_median, train_data$work_readiness_score)
test_data$wr_filled  <- ifelse(is.na(test_data$work_readiness_score),  wr_median, test_data$work_readiness_score)

# Treat citizenship as a category
train_data$sa_citizen_chr <- as.character(train_data$sa_citizen)
test_data$sa_citizen_chr  <- as.character(test_data$sa_citizen)

baseline_model <- glm(employed_status ~ wr_filled + sa_citizen_chr,
                      data = train_data, family = binomial)

summary(baseline_model)



Call:
glm(formula = employed_status ~ wr_filled + sa_citizen_chr, family = binomial, 
    data = train_data)

Coefficients:
                   Estimate Std. Error z value Pr(>|z|)    
(Intercept)         -1.0516     0.5610  -1.875   0.0609 .  
wr_filled            0.9161     0.1061   8.632   <2e-16 ***
sa_citizen_chrTRUE  -0.1441     0.5586  -0.258   0.7964    
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 20801  on 16652  degrees of freedom
Residual deviance: 20726  on 16650  degrees of freedom
  (66 observations deleted due to missingness)
AIC: 20732

Number of Fisher Scoring iterations: 4


Description of all the variables

In [ ]:
variable_check <- data.frame(
  variable = names(train_data),
  type = sapply(train_data, function(x) class(x)[1]),
  unique_values = sapply(train_data, function(x) length(unique(x))),
  missing = sapply(train_data, function(x) sum(is.na(x))),
  missing_percent = sapply(train_data, function(x) mean(is.na(x)) * 100)
)

variable_check

,variable,type,unique_values,missing,missing_percent
,<chr>,<chr>,<int>,<int>,<dbl>
anonymised_id,anonymised_id,character,16719,0,0.00000000
lag_round,lag_round,integer,8,11449,68.47897602
current_round,current_round,integer,8,0,0.00000000
days_since_last_obs,days_since_last_obs,integer,287,11449,68.47897602
total_historical_rounds,total_historical_rounds,integer,5,0,0.00000000
survey_date,survey_date,character,204,0,0.00000000
sample,sample,character,24,0,0.00000000
sample_first,sample_first,character,23,0,0.00000000
employed_status,employed_status,integer,3,66,0.39476045


## 🔮 Step 4: Predict and build a submission
For AUC we submit **probabilities**, not hard labels. The block below writes a valid `submission_baseline.csv`.
Upload it on the competition's Submit page to see your public leaderboard score.

In [ ]:
# Make sure the test set has the same prepared column the model was trained on
test_data$sa_citizen_chr <- as.character(test_data$sa_citizen)

# Predicted PROBABILITY of employment (0-1)
test_prob <- predict(baseline_model, newdata = test_data, type = 'response')

submission <- data.frame(anonymised_id = test_data$anonymised_id,
                         employed_status = test_prob)
write.csv(submission, 'submission_baseline.csv', row.names = FALSE, quote = FALSE)
cat('Saved submission_baseline.csv with', nrow(submission), 'rows\n')
head(submission)


Saved submission_baseline.csv with 3230 rows


,anonymised_id,employed_status
,<chr>,<dbl>
1,ID_10,0.3147682
2,ID_100,0.3147682
3,ID_1000,0.3147682
4,ID_1001,0.3147682
5,ID_1002,0.3147682
6,ID_1003,0.3147682


## 🔄 Step 5: Choose better features and watch the score improve
The baseline features barely helped. Now we swap in features that actually carry signal:
`gender` and `education_level`, while keeping `work_readiness_score`. Same model, better inputs — and the score improves.
This is the core lesson: **which** variables you choose matters more than how many.

In [ ]:
# Fill missing work readiness (recompute to be safe)
wr_median <- median(train_data$work_readiness_score, na.rm = TRUE)
train_data$wr_filled <- ifelse(is.na(train_data$work_readiness_score), wr_median, train_data$work_readiness_score)
test_data$wr_filled  <- ifelse(is.na(test_data$work_readiness_score),  wr_median, test_data$work_readiness_score)

# Align education_level factor levels between train and test
train_data$education_level <- factor(train_data$education_level)
test_data$education_level  <- factor(test_data$education_level, levels = levels(train_data$education_level))

upgraded_model <- glm(employed_status ~ gender + education_level + wr_filled,
                      data = train_data, family = binomial)

up_prob <- predict(upgraded_model, newdata = test_data, type = 'response')
up_prob[is.na(up_prob)] <- mean(train_data$employed_status, na.rm = TRUE)

submission2 <- data.frame(anonymised_id = test_data$anonymised_id, employed_status = up_prob)
write.csv(submission2, 'submission_upgraded.csv', row.names = FALSE, quote = FALSE)
cat('Saved submission_upgraded.csv\n')
head(submission2)


Saved submission_upgraded.csv


,anonymised_id,employed_status
,<chr>,<dbl>
1,ID_10,0.2354129
2,ID_100,0.2354129
3,ID_1000,0.2354129
4,ID_1001,0.2354129
5,ID_1002,0.2354129
6,ID_1003,0.2354129


## 🚀 Over to you — ideas to climb the leaderboard
You just saw that swapping weak features for better ones lifted the score. There is a lot more room to improve:

- **Use participant history:** `employed_lag`, `status_lag`, `status_broad_lag`, `tenure_lag` — a person's *previous* status is a strong clue, when it is present.
- **Handle missingness thoughtfully:** treat 'missing' as its own category; train and test differ in how complete some columns are.
- **More demographics & geography:** `age`, `race`, `province`, `district`, `municipality`.
- **Education & skills:** `school_quintile`, the `matric_*` subjects.
- **Try stronger models:** `ranger` (random forest) or `xgboost`.
- **Always submit probabilities** and validate with AUC, not accuracy.

Good luck!